# RT-DETR Phone Detection Training - T4x2 DDP

This notebook tries to use both Kaggle T4 GPUs with PyTorch DDP.

Use this only after the single-GPU balanced notebook is confirmed to run.

Dataset input:
- Recommended: `mobilephone_wusj2_v7_rtdetr_balanced_kaggle.zip`

Training profile:
- Model: `rtdetr-l.pt`
- Image size: 512
- Epochs: 15
- Device: `0,1`
- Batch retry: 4, then 2

If this hits CUDA OOM, return to the single-GPU balanced notebook.

In [ ]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def ensure_package(package_name, import_name=None):
    import importlib.util
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {package_name} without dependency upgrades...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', package_name])
    else:
        print(f'{package_name} is already available')

ensure_package('ultralytics')

import torch
import yaml
from ultralytics import RTDETR

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))


## 1. Locate dataset and normalize paths

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
RUNS_DIR = WORKING_DIR / 'runs'
RUN_NAME = 'phone_rtdetr_l_balanced_512_t4x2'

yaml_candidates = sorted(INPUT_ROOT.rglob('data.yaml'))
if not yaml_candidates:
    zip_candidates = sorted(INPUT_ROOT.rglob('*.zip'))
    if zip_candidates:
        extract_root = WORKING_DIR / 'extracted_dataset'
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        print('Extracting:', zip_candidates[0])
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(extract_root)
        yaml_candidates = sorted(extract_root.rglob('data.yaml'))

if not yaml_candidates:
    raise FileNotFoundError('No data.yaml found. Add the balanced dataset zip to this notebook input.')

source_yaml = yaml_candidates[0]
dataset_root = source_yaml.parent
print('Dataset root:', dataset_root)

with open(source_yaml, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f) or {}

data_cfg['path'] = str(dataset_root)
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'
data_cfg['test'] = 'test/images'
data_cfg['nc'] = 1
data_cfg['names'] = ['Mobile-phone']

work_yaml = WORKING_DIR / 'mobilephone_rtdetr_balanced_t4x2_data.yaml'
with open(work_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)
print(work_yaml.read_text())


## 2. Dataset sanity check

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_split(split):
    image_dir = dataset_root / split / 'images'
    label_dir = dataset_root / split / 'labels'
    images = [p for p in image_dir.glob('*') if p.suffix.lower() in IMG_EXTS]
    labels = list(label_dir.glob('*.txt')) if label_dir.exists() else []
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    bad = 0
    boxes = 0
    classes = set()
    for label in labels:
        for line in label.read_text(encoding='utf-8', errors='replace').splitlines():
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                bad += 1
                continue
            try:
                cls = int(float(parts[0]))
                vals = list(map(float, parts[1:5]))
                classes.add(cls)
                boxes += 1
                if cls != 0 or any(v < 0 or v > 1 for v in vals) or vals[2] <= 0 or vals[3] <= 0:
                    bad += 1
            except Exception:
                bad += 1
    return {
        'images': len(images),
        'labels': len(labels),
        'missing_labels': len(image_stems - label_stems),
        'extra_labels': len(label_stems - image_stems),
        'boxes': boxes,
        'classes': sorted(classes),
        'bad': bad,
    }

stats = {split: count_split(split) for split in ['train', 'valid', 'test']}
print(stats)
for split, row in stats.items():
    assert row['images'] > 0 and row['labels'] > 0, split
    assert row['missing_labels'] == 0 and row['extra_labels'] == 0, row
    assert row['bad'] == 0, row
    assert row['classes'] == [0], row


## 3. Train RT-DETR-L with T4x2 DDP

In [ ]:
MODEL_NAME = 'rtdetr-l.pt'
EPOCHS = 15
IMGSZ = 512
BATCH_CANDIDATES = [4, 2]
PATIENCE = 6

if torch.cuda.device_count() < 2:
    raise RuntimeError('This T4x2 notebook requires 2 GPUs. Use the single-GPU balanced notebook instead.')
DEVICE = '0,1'

def train_once(batch):
    model = RTDETR(MODEL_NAME)
    return model.train(
        data=str(work_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=batch,
        device=DEVICE,
        workers=4,
        patience=PATIENCE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        optimizer='AdamW',
        lr0=0.0001,
        weight_decay=0.0005,
        cos_lr=True,
        amp=True,
        cache=False,
        deterministic=False,
        mosaic=0.0,
        scale=0.25,
        plots=True,
    )

last_error = None
for batch in BATCH_CANDIDATES:
    try:
        BATCH = batch
        print(f'Training T4x2 RT-DETR-L: imgsz={IMGSZ}, global batch={BATCH}, device={DEVICE}')
        train_results = train_once(BATCH)
        break
    except Exception as exc:
        last_error = exc
        message = str(exc).lower()
        if 'out of memory' not in message and 'childfailed' not in message and 'calledprocesserror' not in message:
            raise
        print(f'DDP failed at batch={batch}: {repr(exc)}')
        print('Retrying smaller batch...')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    raise RuntimeError('T4x2 DDP failed for all batch sizes. Use the single-GPU balanced notebook.') from last_error


## 4. Validate, sample predict, and package

In [ ]:
run_dir = RUNS_DIR / 'detect' / RUN_NAME
best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'
assert best_pt.exists(), f'best.pt not found: {best_pt}'

best_model = RTDETR(str(best_pt))
metrics = best_model.val(
    data=str(work_yaml),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME + '_test_eval',
    exist_ok=True,
    plots=True,
)
print(metrics)

sample_images = sorted((dataset_root / 'test' / 'images').glob('*'))[:24]
best_model.predict(
    source=[str(p) for p in sample_images],
    conf=0.25,
    imgsz=IMGSZ,
    device=0,
    project=str(RUNS_DIR),
    name='rtdetr_t4x2_predict_samples',
    exist_ok=True,
    save=True,
)

package_dir = WORKING_DIR / 'phone_rtdetr_t4x2_export'
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt, package_dir / 'phone_rtdetr_l_best.pt')
if last_pt.exists():
    shutil.copy2(last_pt, package_dir / 'phone_rtdetr_l_last.pt')
shutil.copy2(work_yaml, package_dir / 'mobilephone_rtdetr_t4x2_data.yaml')

for filename in ['results.csv', 'results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png']:
    src = run_dir / filename
    if src.exists():
        shutil.copy2(src, package_dir / filename)

zip_path = shutil.make_archive(str(WORKING_DIR / 'phone_rtdetr_t4x2_training_results'), 'zip', package_dir)
print('Download:', zip_path)
